# 04a | Entity-stance SFT pilot from the financial-domain model

Use the 74-update `domain_v1` checkpoint and original 8K tokenizer from Drive; the first handoff imports weights and creates a new optimizer.
Train on human-confirmed headline stance toward literal entities in the text. This is a private research pilot; targets are not verified security identifiers.
Semantic-event and historical-time audits are incomplete; development metrics do not measure real Twitter or future-market accuracy.
Source articles may carry third-party rights. Keep this material within personal research and preserve source notices.

Select T4 and run cells in order. Upload local `entity_v1_upload.zip` initially; subsequent runs reuse the Drive backup.
After disconnection, confirm the previous process has stopped before rerunning from the beginning. Record the 40-character commit shown in the first cell; if main changes,
set REF to that original full commit before resuming. Do not start two training processes concurrently.

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # Resolve and display the commit first; use the full 40-character SHA for reproduction.
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF does not resolve uniquely; enter the full commit SHA.")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM belongs to another repository; choose a new directory.")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("Pinned code revision:", REVISION)

# Keep the bundled Colab CUDA / PyTorch; install only the other project dependencies.
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} is incompatible with the existing environment:{incompatible}")
    print("tokenizers compatibility check passed:", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"Missing {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} does not satisfy {requirement}")
    if problems:
        raise RuntimeError("Project dependency check failed:" + "; ".join(problems))
    print("Installed project dependencies passed validation.")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"Project Python source is missing:{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"The notebook imported the wrong project location:{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("The checkpoint restore interface is unavailable in this notebook.")
    print("Current notebook kernel import check passed:", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("Fresh Python process import check passed:", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("PyTorch >= 2.5 is required; select a newer Colab runtime.")
tokenizer_requirements = tokenizer_install_requirements()
print("Active tokenizers constraints:", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("The PyTorch environment changed unexpectedly; preserve the output and stop training.")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook setup complete; PyTorch remains:", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Stage inputs are missing; prepare them or restore them from Drive first:\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"Free disk space in /content:{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("This stage requires existing seed inputs and versioned backups; mount Drive first.")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive project directory:", DRIVE_ROOT)

In [ ]:
from finllm.sft_pilot_runtime import install_task_zip, prepare_sft_pilot, run_sft_pilot, backup_sft_pilot
from finllm.expansion_runtime import _inventory
TASK_VERSION = "entity_v1"
RUN_VERSION = "sft_entity_v1"
TASK_ZIP_SHA256 = "e87d12013271eb8f93f926b0b2c71cef15b0c777cc58d6e08348773dbd6a7be6"
TASK_DRIVE = DRIVE_ROOT / "tasks" / TASK_VERSION
if not (TASK_DRIVE / "SNAPSHOT_COMPLETE.json").exists():
    from google.colab import files
    import tempfile
    with tempfile.TemporaryDirectory(prefix="finllm-upload-") as folder:
        original_cwd = Path.cwd()
        try:
            os.chdir(folder)
            print("Select local entity_v1_upload.zip; it will be verified and backed up to Drive.")
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError("Select exactly one task ZIP.")
            name = next(iter(uploaded))
            if Path(name).name != name:
                raise ValueError("Invalid upload filename.")
            result = install_task_zip(Path(folder) / name, DRIVE_ROOT, TASK_ZIP_SHA256, TASK_VERSION)
        finally:
            os.chdir(original_cwd)
    print("Task-data backup:", json.dumps(result, indent=2, ensure_ascii=False))
else:
    _inventory(TASK_DRIVE)
    print("Task-data snapshot reused and verified:", TASK_DRIVE)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select T4 GPU, then rerun from the first cell.")
pilot = prepare_sft_pilot(PROJECT, DRIVE_ROOT, task_version=TASK_VERSION, run_version=RUN_VERSION)
print("Verified domain parent model:", json.dumps(pilot["parent_audit"], indent=2, ensure_ascii=False))
print("Task partition counts:", json.dumps(pilot["counts"], indent=2, ensure_ascii=False))
print("Current GPU:", torch.cuda.get_device_name(0))

In [ ]:
SFT_PROBE_SUCCEEDED = False
SFT_TRAIN_SUCCEEDED = False
SFT_BACKED_UP = False
probe = run_sft_pilot(pilot, "probe", finllm)
print(json.dumps(probe, indent=2, ensure_ascii=False))
SFT_PROBE_SUCCEEDED = True

In [ ]:
SFT_TRAIN_SUCCEEDED = False
SFT_BACKED_UP = False
if not globals().get("SFT_PROBE_SUCCEEDED", False):
    raise RuntimeError("Complete the one-step SFT verification first.")
result = run_sft_pilot(pilot, "continue", finllm)
print(json.dumps(result, indent=2, ensure_ascii=False))
SFT_TRAIN_SUCCEEDED = True

In [ ]:
SFT_EVAL_SUCCEEDED = False
if not globals().get("SFT_TRAIN_SUCCEEDED", False):
    raise RuntimeError("Complete SFT training first.")
SFT_OUTPUT = Path(pilot["config"]["output_dir"])
SFT_DEV = pilot["task_local"] / "development.jsonl"
for name, checkpoint in (("domain_before", Path(pilot["config"]["init_checkpoint"])),
                         ("sft_best", SFT_OUTPUT / "best.pt")):
    finllm("evaluate", "--checkpoint", checkpoint, "--tokenizer", pilot["config"]["tokenizer"],
           "--input", SFT_DEV, "--output", SFT_OUTPUT / (name + "_development.json"),
           "--target-mode", "literal_entity")
SFT_EVAL_SUCCEEDED = True
print("Saved evaluations of the domain model and best SFT checkpoint on the same development set.")

In [ ]:
SFT_BACKED_UP = False
if not globals().get("SFT_TRAIN_SUCCEEDED", False) or not globals().get("SFT_EVAL_SUCCEEDED", False):
    raise RuntimeError("SFT training or the development comparison is incomplete; preserve existing checkpoints.")
backup = backup_sft_pilot(pilot)
print(json.dumps(backup, indent=2, ensure_ascii=False))
SFT_BACKED_UP = True
print("Entity pilot complete and backed up:", pilot["stage_drive"] / "run")

Keep training reports, development macro-F1 and `verified: true` backup output.
Small-sample results may vary. This notebook does not complete final testing, asset-identifier mapping, DPO preference review, calibration or real X evaluation.